In [1]:
from temisUv import *

def GetDict(lon, lat, ncFile):

    ilon = getIlon(lon)
    ilat = getIlat(lat)
    
    datas = {"longitude": lon,
            "latitude": lat,
            "infos": []}

    with netCDF4.Dataset(ncFile,'r') as nc:

        dates = nc.groups['PRODUCT'].variables['date'][:]
        nDays = len(dates)

        uvief = nc.groups['PRODUCT'].variables['uvi_clear'][:,ilat,ilon]
        ozone = nc.groups['PRODUCT'].variables['ozone_column'][:,ilat,ilon]

    for iDay in range(0,nDays,1):

        datas["infos"].append({"date": dates[iDay],
                               "uv_index": uvief[iDay],
                               "ozone": ozone[iDay]})
        
    return datas

In [2]:
import requests
from bs4 import BeautifulSoup

url = "https://www.temis.nl/uvradiation/nrt/uvindex.php"

def get_first_nc_link(url):
    """Récupère le premier lien contenant un fichier .nc depuis une URL donnée."""
    try:
        soup = BeautifulSoup(requests.get(url).text, 'html.parser')
        for link in soup.find_all('a', href=True):
            if link['href'].endswith('.nc'):
                return requests.compat.urljoin(url, link['href'])
    except Exception as e:
        print(f"Erreur : {e}")
    return None

In [3]:
def download(path, url):
    try:
        # Télécharger le fichier
        response = requests.get(url)
        response.raise_for_status()  # Vérifie les erreurs HTTP
        
        # Enregistrer le contenu dans un fichier local
        with open(path, "wb") as fichier:
            fichier.write(response.content)

    except requests.exceptions.RequestException as e:
        print(f"Erreur lors du téléchargement du fichier : {e}")
    except Exception as e:
        print(f"Erreur lors de l'enregistrement du fichier : {e}")

In [4]:
import os 
from datetime import date



# Telecharge si n'existe pas. 

date_du_jour = date.today().strftime("%Y%m%d")
dossier_courant = os.getcwd()

dossier = os.path.join(dossier_courant, 'DOWNLOAD' ,date_du_jour)
path_nc = os.path.join(dossier, 'uvief_fc.nc')

if not os.path.exists(path_nc):
    try:
        os.makedirs(dossier)
    except:
        pass
    
    # Fichier nc du jour
    nc_link = get_first_nc_link(url)
    download(path_nc, nc_link)

print(f"Premier lien .nc trouvé : {nc_link}" if nc_link else "Aucun lien .nc trouvé.")

Premier lien .nc trouvé : https://d1qb6yzwaaq4he.cloudfront.net/uvradiation/UVI/uvief_fc.nc


In [5]:
# Exemple Genève
lon, lat = 6.132936441, 46.220473615

GetDict(lon, lat, path_nc)

{'longitude': 6.132936441,
 'latitude': 46.220473615,
 'infos': [{'date': 20250113, 'uv_index': 0.858, 'ozone': 349.6},
  {'date': 20250114, 'uv_index': 0.965, 'ozone': 313.1},
  {'date': 20250115, 'uv_index': 0.994, 'ozone': 308.8},
  {'date': 20250116, 'uv_index': 0.951, 'ozone': 329.4},
  {'date': 20250117, 'uv_index': 0.998, 'ozone': 319.2},
  {'date': 20250118, 'uv_index': 0.974, 'ozone': 333.9},
  {'date': 20250119, 'uv_index': 0.996, 'ozone': 332.7},
  {'date': 20250120, 'uv_index': 0.982, 'ozone': 344.7},
  {'date': 20250121, 'uv_index': 0.942, 'ozone': 368.6}]}

In [3]:
from app import custom

# Exemple Genève
answer = custom.request_uv_week(lon=6.132936441, lat=46.220473615)

display(answer)

{'longitude': 6.132936441,
 'latitude': 46.220473615,
 'infos': [{'date': 20250113, 'uv_index': 0.858, 'ozone': 349.6},
  {'date': 20250114, 'uv_index': 0.965, 'ozone': 313.1},
  {'date': 20250115, 'uv_index': 0.994, 'ozone': 308.8},
  {'date': 20250116, 'uv_index': 0.951, 'ozone': 329.4},
  {'date': 20250117, 'uv_index': 0.998, 'ozone': 319.2},
  {'date': 20250118, 'uv_index': 0.974, 'ozone': 333.9},
  {'date': 20250119, 'uv_index': 0.996, 'ozone': 332.7},
  {'date': 20250120, 'uv_index': 0.982, 'ozone': 344.7},
  {'date': 20250121, 'uv_index': 0.942, 'ozone': 368.6}]}